In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#Pulling from Silver layer

In [0]:
df_main = spark.table('openfoodfactslakehouse.silver.products')

#Table columns

In [0]:
fact_product_entries_cols = [
    "entry_id",
    "product_id",
    "brand_id",
    "entry_creator",
    "product_data_completeness",
    "product_creation_datetime",
    "last_user_modification_datetime",
    "last_system_update_datetime",
    "last_product_image_datetime"
]

dim_brands_cols = [
    "brand_id",
    "brands_list",
    "brand_owner"
]

dim_products_cols = [
    "product_id",
    "product_page_url",
    "product_name",
    "packaging_type",
    "packaging_details",
    "manufacturing_locations",
    "purchasing_locations",
    "countries",
    "ingridients_description",
    "allergens",
    "serving_quantity",
    "nova_group",
    "pnns_groups_2",
    "environmental_score_score",
    "environmental_score_grade",
    "product_weight",
    "unique_scan_count",
    "product_image_url",
    "image_ingredients_url",
    "image_nutrition_data_url",
    "energy_kj_100g",
    "energy_kcal_100g",
    "fat_100g",
    "saturated_fat_100g",
    "cholesterol_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "lactose_100g",
    "starch_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "alcohol_100g",
    "vitamin_a_100g",
    "vitamin_d_100g",
    "vitamin_e_100g",
    "vitamin_k_100g",
    "vitamin_c_100g",
    "vitamin_b1_100g",
    "vitamin_b2_100g",
    "vitamin_b6_100g",
    "vitamin_b9_100g",
    "vitamin_b12_100g",
    "potassium_100g",
    "calcium_100g",
    "phosphorus_100g",
    "iron_100g",
    "magnesium_100g",
    "zinc_100g",
    "copper_100g",
    "iodine_100g",
    "caffeine_100g",
    "carbon_footprint_100g"
]

#Splitting table into 3

In [0]:
w_entry_id = Window.orderBy(
  'product_id'
)

w_brand_id = Window.orderBy(
  'brands_list',
)

df_main = df_main.withColumn(
  'entry_id',
  F.row_number().over(w_entry_id)
)

df_main = df_main.withColumn(
  'brand_id',
  F.dense_rank().over(w_brand_id)
)

fact_product_entries = df_main.select(*fact_product_entries_cols)
dim_brands = df_main.select(*dim_brands_cols)
dim_products = df_main.select(*dim_products_cols)

fact_product_entries = fact_product_entries.distinct()
dim_brands = dim_brands.distinct()
dim_products = dim_products.distinct()

window = Window.partitionBy("brand_id", "brands_list") \
      .orderBy(
          F.when((F.col("brand_owner").isNull()) |
        (F.col("brand_owner") == "N/A"), 1).otherwise(0)
      )

dim_brands = dim_brands.withColumn(
  'rn',
  F.row_number() \
    .over(window)
).filter(F.col('rn') == 1).drop('rn')

#Writing to Gold Layer

In [0]:
df_dict = {
    "fact_product_entries": fact_product_entries,
    "dim_brands": dim_brands,
    "dim_products": dim_products
}

for name, df in df_dict.items():
    (
        df.write
        .mode('overwrite')
        .format('delta')
        .option('mergeSchema', 'true')
        .saveAsTable(f"openfoodfactslakehouse.gold.{name}")
    )